# Basket

## Imports et Scripts originaux

In [1]:
import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
import statsmodels.api as sm
from joblib import dump, load

In [48]:
def score_classifier(dataset,classifier,labels):

    """
    performs 3 random trainings/tests to build a confusion matrix and prints results with precision and recall scores
    :param dataset: the dataset to work on
    :param classifier: the classifier to use
    :param labels: the labels used for training and validation
    :return:
    """

    kf = KFold(n_splits=3,random_state=50,shuffle=True)
    confusion_mat = np.zeros((2,2))
    recall = 0
    for training_ids,test_ids in kf.split(dataset):
        training_set = dataset[training_ids]
        training_labels = labels[training_ids]
        test_set = dataset[test_ids]
        test_labels = labels[test_ids]
        classifier.fit(training_set,training_labels)
        predicted_labels = classifier.predict(test_set)
        confusion_mat+=confusion_matrix(test_labels,predicted_labels)
        recall += recall_score(test_labels, predicted_labels)
    recall/=3
    print(confusion_mat)
    print(recall)
    return recall #pour la recherche

J'ai ajouté un return pour mes tests sur des classificateurs\
J'ai choisi de ne pas changer le reste de la fonction de scoring, mais ultérieurement je reviens sur les évaluations de performance de classificateurs pour mon choix final.

In [49]:
# Load dataset
df = pd.read_csv(".\\nba_logreg.csv")

# extract names, labels, features names and values
names = df['Name'].values.tolist() # players names
labels = df['TARGET_5Yrs'].values # labels
paramset = list(df.drop(['TARGET_5Yrs','Name'],axis=1).columns.values)
df_vals = df.drop(['TARGET_5Yrs','Name'],axis=1).values
# type(df_vals)

# replacing Nan values (only present when no 3 points attempts have been performed by a player)
for x in np.argwhere(np.isnan(df_vals)):
    df_vals[x]=0.0

# normalize dataset
X = MinMaxScaler().fit_transform(df_vals)

#example of scoring with support vector classifier
score_classifier(X,SVC(),labels)



[[271. 238.]
 [145. 686.]]
0.82551959002102


0.82551959002102

C'est la fin du code d'origine fourni pour le problème.\
On a le dataset découpé et une fonction de scoring pour évaluer nos classificateurs.\
L'objectif est de trouver le classificateur qui répond le mieux à la demande des investisseurs. (Rappel : L’objectif est de fournir un classifier permettant de prédire qu’un joueur vaut le coup d’investir sur lui car il va durer plus de 5 ans en NBA).

Pour atteindre cet objectif, cela vaut la peine de s'intéresser à la mesure de performance que l'on souhaite optimiser pour répondre à la demande de l'investisseur.\
Si on optimise le recall, on cherche à s'assurer que parmi les "bons" joueurs, le plus possible sont sélectionnés par notre classificateur, c'est à dire qu'on ne passe pas à coté de bons joueur.\
Si on optimise la précision (accuracy), on cherche à ce que sur les prédictions du modèle, on se trompe le moins souvent possible, on n'investit pas dans les mauvais joueurs et on investit sur les bons joueurs.\
Il n'y a pas de choix optimal pour toutes les situations :\
Si les investisseurs ne veulent pas passer à coté de talents, mais que ce n'est pas un gros problème d'investir dans un mauvais joueur, alors c'est le recall qu'on va regarder en priorité.\
Si les investisseurs ne veulent pas "miser sur un mauvais cheval", et qu'ils n'ont pas de problème s'ils passent à coté de bons joueurs (limiter les risques, mais avoir des coûts d'opportunité), il faut s'intéresser à la précision (precision en anglais, pas accuracy).\
S'ils souhaitent une approche plus équilibrées, il y a toujours l'accuracy, le F1 score, le score AUC.\
Pour ma recherche, je m'intéresse au recall (parce que c'est la direction suggérée par le script original), mais aussi je regarde le F1 score et l'accuracy.

## Travail sur les classificateurs

Dans une première approche plus personnelle du problème, j'ai travaillé sur le dataset et les variables (ExploData), mais cela ne s'intégrait pas très bien dans la consigne. Donc j'ai repris le dataset d'origine et travaillé sur une gamme de classificateurs et leurs hyperparamètres.\
J'affiche ici les résultats de ma recherche pour une sélection de classificateurs.

In [50]:
# TODO build a training set and choose a classifier which maximize recall score returned by the score_classifier function
# X, labels

### Training Set

En deuxieme approche, je pense qu'une étape de préprocessing et de construction de meilleures features est souhaitable.\
Pour résumer mon analyse du Dataset, il y a des lignes dupliquées, les variables sont aussi très correllées. Certaines opérations sur les variables permettent de corriger cela (cf. ExploData).\
Aussi, à cette étape, ou pourrait retirer certaines features après une sélection de features.

In [51]:
# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42, stratify=labels)


### Recherche de Classificateur

J'ai exclu de ma recherche les modèles Réseaux de Neurones pour des raisons de temps et de ressources. Aussi, cela ne me semblait pas pertinent pour le contexte.

#### Dummy Classifier

D'après mes observations et essais, le problème principal pour obtenir un classificateur performant sur ce jeu de données, c'est que les variables ne semblent pas expliquer une grande partie de la variable à prédire en un sens.\
Mon idée est qu'une grande partie des raison qu'un joueur va durer 5 ans en NBA n'est lié à ses performances précédentes que dans une mesure limitée. Il est plus possible que beaucoup de hasard (ou des variables non présentes dans le dataset a minima) déterminent la longévité en NBA. (Plus à ce sujet avec la Régression logistique).\
\
Le piège est que pour optimiser le recall, il existe une solution triviale (d'intérêt limité) qu'on retrouve dans les autres classificateurs.\
Il suffit d'investir sur tous les joueurs et on est sûrs qu'on ne passe pas à côté d'un bon joueur.\
Etant donné que les classes sont déséquilibrées (il y a plus de bons joueurs que de mauvais), on arrive sur ce résultat avec le Dummy Classifier (en attribuant systématiquement la classe dominante).\
Pourquoi mentionner ce cas? Le déséquilibre de classes implique que ce cas se reproduit dès qu'on choisit un modèle à fort biais (SVM, certaines variantes de Forêts ou d'arbres de classification). Ces modèles vont "underfitter les données", ce n'est pas optimal, et un investisseur ne va pas nous aimer si on lui dit de simplement investir sur chaque joueur.


In [52]:
classifier = DummyClassifier(strategy="most_frequent")

def tester(classifier):
    classifier.fit(X_train,y_train)
    y_pred=classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print("accuracy sur le test : ",accuracy)
    print("recall sur le test : ",recall)
    print("f1 score sur le test : ",f1)

tester(classifier)
print("")
score_classifier(X_train,classifier,y_train)

accuracy sur le test :  0.6194029850746269
recall sur le test :  1.0
f1 score sur le test :  0.7649769585253456

[[  0. 407.]
 [  0. 665.]]
1.0


1.0

On a le meilleur recall possible!

#### SVM

In [53]:
classifier = SVC()
tester(classifier)
print("")
score_classifier(X_train,classifier,y_train)

accuracy sur le test :  0.7164179104477612
recall sur le test :  0.8313253012048193
f1 score sur le test :  0.7840909090909091

[[223. 184.]
 [130. 535.]]
0.8059044885888298


0.8059044885888298

C'est l'exemple donné et ce n'est pas le pire classificateur pour notre jeu de données. L'accuracy est faible mais le recall est plutôt bon (pour notre jeu de données).

#### Random Forest

On fait une recherche sur les hyperparamètres pour optimiser le recall.

In [54]:
# Liste des hyperparamètres à tester
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

best_score = -float("inf")
best_params = None

# Exploration de paramètres
for n in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        for min_samples in param_grid['min_samples_split']:
            classifier = RandomForestClassifier(n_estimators=n, max_depth=depth, min_samples_split=min_samples, random_state=42)
            
            score = score_classifier(X_train, classifier, y_train) 
            
            # Affichage du score pour chaque combinaison
            print(f"n_estimators={n}, max_depth={depth}, min_samples_split={min_samples} --> Score: {score:.4f}")
            
            if score > best_score:
                best_score = score
                best_params = {'n_estimators': n, 'max_depth': depth, 'min_samples_split': min_samples}


[[204. 203.]
 [159. 506.]]
0.7613141714218764
n_estimators=50, max_depth=None, min_samples_split=2 --> Score: 0.7613
[[196. 211.]
 [135. 530.]]
0.7972828055678097
n_estimators=50, max_depth=None, min_samples_split=5 --> Score: 0.7973
[[209. 198.]
 [147. 518.]]
0.7797795502849355
n_estimators=50, max_depth=None, min_samples_split=10 --> Score: 0.7798
[[195. 212.]
 [138. 527.]]
0.7927579186899817
n_estimators=50, max_depth=10, min_samples_split=2 --> Score: 0.7928
[[209. 198.]
 [144. 521.]]
0.7839866108300243
n_estimators=50, max_depth=10, min_samples_split=5 --> Score: 0.7840
[[205. 202.]
 [141. 524.]]
0.7885508581448929
n_estimators=50, max_depth=10, min_samples_split=10 --> Score: 0.7886
[[201. 206.]
 [162. 503.]]
0.7567892845440484
n_estimators=50, max_depth=20, min_samples_split=2 --> Score: 0.7568
[[199. 208.]
 [137. 528.]]
0.7943968026818068
n_estimators=50, max_depth=20, min_samples_split=5 --> Score: 0.7944
[[209. 198.]
 [147. 518.]]
0.7797795502849355
n_estimators=50, max_depth

In [21]:
# Résultat final
print("\nMeilleurs paramètres :", best_params)
# n_estimators=50, max_depth=None, min_samples_split=5
print("Meilleur score :", best_score)
# 0.7972
score_classifier(X,RandomForestClassifier(random_state=42),labels)
# 0.7951


Meilleurs paramètres : {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 5}
Meilleur score : 0.7972828055678097
[[259. 250.]
 [170. 661.]]
0.7951011894631627


0.7951011894631627

In [22]:
classifier = RandomForestClassifier(n_estimators=50, max_depth=None, min_samples_split=5, random_state=42)
tester(classifier)

accuracy sur le test :  0.7126865671641791
recall sur le test :  0.8373493975903614
f1 score sur le test :  0.7830985915492957


Les performances sont comparables au SVM.

#### Gradient Boosting

In [23]:
# Liste des hyperparamètres à tester
param_grid = {
    'n_estimators': [50, 100, 200],      # Nombre d'arbres
    'max_depth': [3, 6, 10],             # Profondeur max des arbres
    'learning_rate': [0.01, 0.1, 0.2],   # Taux d'apprentissage
    'subsample': [0.8, 1.0]              # Pourcentage d'échantillons utilisés par arbre
}
best_score = -float("inf")
best_params = None

# Exploration de paramètres
for n in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        for lr in param_grid['learning_rate']:
            for subsample in param_grid['subsample']:
                classifier = GradientBoostingClassifier(n_estimators=n, max_depth=depth, learning_rate=lr, subsample=subsample, random_state=42)
                
                score = score_classifier(X_train, classifier, y_train) 
                
                # Affichage du score pour chaque combinaison
                print(f"n_estimators={n}, max_depth={depth}, learning_rate={lr}, subsample={subsample} --> Score: {score:.4f}")
                
                if score > best_score:
                    best_score = score
                    best_params = {'n_estimators': n, 'max_depth': depth, 'learning_rate': lr, 'subsample': subsample}

[[ 61. 346.]
 [ 29. 636.]]
0.9568682261308608
n_estimators=50, max_depth=3, learning_rate=0.01, subsample=0.8 --> Score: 0.9569
[[ 69. 338.]
 [ 41. 624.]]
0.9382501876288124
n_estimators=50, max_depth=3, learning_rate=0.01, subsample=1.0 --> Score: 0.9383
[[204. 203.]
 [134. 531.]]
0.7987824566117855
n_estimators=50, max_depth=3, learning_rate=0.1, subsample=0.8 --> Score: 0.7988
[[200. 207.]
 [144. 521.]]
0.7837734391172667
n_estimators=50, max_depth=3, learning_rate=0.1, subsample=1.0 --> Score: 0.7838
[[200. 207.]
 [153. 512.]]
0.7709083699141694
n_estimators=50, max_depth=3, learning_rate=0.2, subsample=0.8 --> Score: 0.7709
[[200. 207.]
 [147. 518.]]
0.7790219538355413
n_estimators=50, max_depth=3, learning_rate=0.2, subsample=1.0 --> Score: 0.7790
[[ 74. 333.]
 [ 44. 621.]]
0.9343350196705623
n_estimators=50, max_depth=6, learning_rate=0.01, subsample=0.8 --> Score: 0.9343
[[106. 301.]
 [ 73. 592.]]
0.890785546957875
n_estimators=50, max_depth=6, learning_rate=0.01, subsample=1.0

In [24]:
# Résultat final
print("\nMeilleurs paramètres :", best_params)
# n_estimators=50, max_depth=3, learning_rate=0.01, subsample:0.8
print("Meilleur score :", best_score)
# 0.9568
score_classifier(X,GradientBoostingClassifier(random_state=42),labels)
# 0.7916


Meilleurs paramètres : {'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8}
Meilleur score : 0.9568682261308608
[[276. 233.]
 [173. 658.]]
0.7916854614698376


0.7916854614698376

In [25]:
classifier = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=0.01, subsample=0.8, random_state=42)
tester(classifier)

accuracy sur le test :  0.6380597014925373
recall sur le test :  0.9879518072289156
f1 score sur le test :  0.7717647058823529


Ici, on a un exemple du problème Biais/Variance mentionné au niveau du Dummy classifier.\
On arrive avec le plus faible nombre d'estimateurs, le plus faible learning rate, la plus faible profondeur d'arbre. C'est au final le classificateur qui va le moins fitter sur la donnée et se rapproche le plus du Dummy classifier.\
Les performances sont par conséquent quasiment identiques.

#### XGBoost

Les essais avec XGBoost sont très similaires au cas précédent. Les conclusions sont les mêmes

In [26]:
classifier = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.01, subsample=0.8, random_state=42)
tester(classifier)

accuracy sur le test :  0.6268656716417911
recall sur le test :  0.9879518072289156
f1 score sur le test :  0.7663551401869159


#### Régression Logistique

J'ai gardé la régression logistique pour la fin car il y a quelques atouts intéressants par rapport aux autres classificateurs. Quand on parle de régression logistique, on peut s'intéresser à la significativité des coefficients, et on connait les coefficients pour chaque variable du jeu d'apprentissage.\
\
Les performances ne sont pas meilleures concernant le recall que d'autres classificateurs, mais l'explicatibilité est un atout qui peut faire qu'on le favorise à d'autres pour notre problème. (Et on a vu que si on souhaite uniquement le meilleur recall, les modèles qu'on sélectionne sont très biaisés.)

In [27]:
#Regression Logistique
classifier = LogisticRegression()
tester(classifier)

accuracy sur le test :  0.7238805970149254
recall sur le test :  0.8554216867469879
f1 score sur le test :  0.7932960893854749


In [35]:
classifier = LogisticRegression()
hist = classifier.fit(X_train,y_train)
print(classifier.intercept_)
pd.DataFrame(classifier.coef_.reshape(1,-1),columns=paramset)

[-2.22619357]


,GP,MIN,PTS,FGM,FGA,FG%,3P Made,3PA,3P%,FTM,FTA,FT%,OREB,DREB,REB,AST,STL,BLK,TOV
0,2.418939,-0.232098,0.601308,0.555389,-0.053144,0.423445,0.479622,-0.741478,0.349315,0.668238,0.285942,-0.152219,1.491099,-0.00371,0.575084,0.724961,0.007333,0.8563,-0.122096


J'utilise statsmodels pour afficher les statistiques sur le modèle Régression logistique (ce n'est pas vraiment le meme modèle donc mais ça donne une idée sur la significativité des coefficients)

In [55]:
#logreg_stats = sm.Logit(y_train, X_train).fit()
print(paramset)
#X_train1 = pd.DataFrame(X_train, columns=paramset)
X_train1 = pd.DataFrame(sm.add_constant(X_train), columns=['Intercept'] + paramset)
model = sm.Logit(y_train,X_train1)
stats = model.fit()
print(stats.summary())

['GP', 'MIN', 'PTS', 'FGM', 'FGA', 'FG%', '3P Made', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV']
Optimization terminated successfully.
         Current function value: 0.552565
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                 1072
Model:                          Logit   Df Residuals:                     1052
Method:                           MLE   Df Model:                           19
Date:                Tue, 11 Mar 2025   Pseudo R-squ.:                  0.1677
Time:                        16:11:28   Log-Likelihood:                -592.35
converged:                       True   LL-Null:                       -711.70
Covariance Type:            nonrobust   LLR p-value:                 5.958e-40
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------

Ce ne sont pas les même coefficients (statsmodels et Scikit-learn ont des méthodes différentes pour estimer les coefficients).\
Cependant on remarque que les significativités sont faibles pour beaucoup, avec une grande partie des p-valeurs qui sont au dessus de 10%.\
\
Le modèle n'est pas de bonne qualité (statistiquement parlant, aussi le pseudo-R² est de 0.16, loin de 1). Pour remédier à ce problème, il faudrait travailler le dataset, décorréler nos features, voire même construire de nouvelles features. On peut aussi regarder les résultats obtenus en faisant une Analyse en Composantes principales sur le dataset et en appliquant nos classificateurs dans cet espace.\
\
Dans mon premier notebook (ExploData), j'avais commencé ce travail mais je n'étais pas sûr que cela était vraiment respectueux de la consigne.

### Conclusion de la Recherche de Classificateur

Au sortir de la comparaison de ces différents modèles, on a un aperçu de leur performance dans le cadre de l'étude. La consigne d'origine d'optimiser le recall s'est avérée ne pas suffire à obtenir un model satisfaisant (en tout cas dans le contexte d'un investisseur). Si on devait choisir un modèle, il faudrait plus d'information sur l'investisseur et son besoin, cela permet de déterminer quelle mesure de performance prioriser pour notre sélection.\
\
Un travail sur les données (construction de features, vérification des données - suppession des duplicatas, ACP), peut permettre d'obtenir un classificateur plus performant (et consistant, pertinent et interprétable).\
Ce travail permettrait d'intégrer une étape de sélection de features.\
\
Pour la suite de l'exercie, je choisis d'utiliser la régression logistique, en tant qu'exemple pour la partie API.

In [59]:
modele1 = LogisticRegression()
modele1.fit(X_train,y_train)
dump(modele1, "modele.joblib")
modele2 = load("modele.joblib")
modele2.predict(X_test)
np.save("X_test.npy", X_test)